In [ ]:
from sentence_transformers import SentenceTransformer
import json
import numpy as np
import os

path = '../data/processed/chunks_peps.json'
outpath_embeddings = '../data/processed/embeddings_peps.npy'
caminho_json = "../data/processed/chunks_peps.json" # Ler o arquivo JSON

# Definir o modelo 
model = SentenceTransformer('BAAI/bge-small-en-v1.5') # as consultas precisam estar em ingles para manter a perfomance
# model = SentenceTransformer('intfloat/multilingual-e5-small') # multi idiomas

Loading weights:   0%|          | 0/199 [00:02<?, ?it/s]

In [17]:
with open(caminho_json, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

# Extrair apenas o texto de cada chunk para uma lista
textos = [chunk['text'] for chunk in chunks]

In [18]:
if os.path.exists(outpath_embeddings):
    # Se o arquivo .npy já existe, carrega do disco em milissegundos
    embeddings = np.load(outpath_embeddings)
    print("✅ Embeddings existentes carregados com sucesso!")

else:
    # Se não existe, inicializa o modelo e gera os embeddings
    print("⚡ Gerando embeddings (arquivo não encontrado)...")

    embeddings = model.encode(
        textos, 
        show_progress_bar=True
    )
    
    # Salva na pasta ../data/processed/
    np.save(outpath_embeddings, embeddings)
    print("✅ Embeddings gerados e salvos com sucesso!")

✅ Embeddings existentes carregados com sucesso!


In [19]:
import faiss

In [20]:
class busca:
    def __init__(self, model, vetor, k, xq):
        '''Indique o embendding, modelo, a quantidade de caracteres correspondente e também a query'''
        self.model = model
        self.vetor = np.ascontiguousarray(vetor, dtype=np.float32) # Garante que os vetores estejam no formato float32 que o FAISS exige
        self.k = k
        
        #garantir que o vetor da query seja float32
        vetor_query = model.encode([xq]) 
        self.xq = np.ascontiguousarray(vetor_query, dtype=np.float32)

        self.d = self.vetor.shape[1]
        # self.index = None # armazenar o indice

        self.nlist = 50  # defini a quantiade de celulas a dividir toda base de embendding
        self.m = 8 # quantidade de sub vetores
        self.bits = 8 # quantidade de bits para guardar

    def treino(self, index): # verificar se está treinado
        self.index = index

        if self.index.is_trained:
            pass
        else:
            index.train(self.vetor)  

        return index

    def retorno(self, index):
        index = self.treino(index)      
        index.add(self.vetor)
        self.index = index
        
        D, I = index.search(self.xq, self.k)  # busca
        print(I)

        return I

    def metrica(self, tipo):
        '''Indique o tipo de busca: 1 para IndexFlatL2, 2 para XXXX e 3 para YYYY'''
        self.tipo = tipo

        if self.tipo == 1: #IndexFlatL2
            index = faiss.IndexFlatL2(self.d)

            return self.retorno(index)

        elif self.tipo == 2: #IndexIVFFlat
            self.quantificador = faiss.IndexFlatL2(self.d) # mapeear e indentificar qual celula o vetor esta

            index = faiss.IndexIVFFlat(
                self.quantificador, #
                self.d, 
                self.nlist
            )

            return self.retorno(index)

        elif self.tipo == 3: #IndexIVFPQ
            self.quantificador = faiss.IndexFlatL2(self.d) # mapeear e indentificar qual celula o vetor esta

            index = faiss.IndexIVFPQ(
                self.quantificador, 
                self.d, 
                self.nlist, # quantidade de divisao
                self.m, # quantidade de sub vetores
                self.bits # quantidade de bits para guardar
            ) 

            return self.retorno(index)

        else:
           print('Indicar a métrica é obrigatório')
           return None


In [21]:
flat = busca(
    model, 
    embeddings,
    4,
    "How many spaces should be used for indentation in Python code according to PEP 8?"
)

In [22]:
%%time
I_flatL2 = flat.metrica(1)

[[5648 4256 8421 4566]]
CPU times: user 24.6 ms, sys: 47.1 ms, total: 71.8 ms
Wall time: 167 ms


In [23]:
# Pega a lista de índices da query
indices_retornados = I_flatL2[0]

# Mapeia diretamente na sua lista de chunks lida do JSON
chunks_encontrados = [textos[idx] for idx in indices_retornados if idx != -1]

# Imprime cada resultado em uma única linha limpa
for i, texto in enumerate(chunks_encontrados, 1):
    texto_linha_unica = texto.replace('\n', ' ')
    print(f"[{i}] {texto_linha_unica}")

[1] scheme of simple nor composite statements but rather establish a category of its own. - *Putting the expression on a separate line after "match".* The idea is to use the expression yielding the subject as a statement to avoid the singularity of ``match`` having no actual block despite the colons:: match: expression case pattern_1: ... case pattern_2: ... This was ultimately rejected because the first block would be another novelty in Python's grammar: a block whose only content is a single expression rather than a sequence of statements. Attempts to amend this issue by adding or repurposing yet another keyword along the lines of ``match: return expression`` did not yield any satisfactory solution. Although flat indentation would save some horizontal space, the cost of increased complexity or unusual rules is too high. It would also complicate life for simple-minded code editors. Finally, the horizontal space issue can be alleviated by allowing "half-indent" (i.e. two spaces instead

In [24]:
%%time
I_flatIVXflat = flat.metrica(2)

[[4256 4566 4220 8453]]
CPU times: user 833 ms, sys: 125 ms, total: 958 ms
Wall time: 895 ms


In [25]:
# Pega a lista de índices da query
indices_retornados = I_flatIVXflat[0]

# Mapeia diretamente na sua lista de chunks lida do JSON
chunks_encontrados = [textos[idx] for idx in indices_retornados if idx != -1]

# Imprime cada resultado em uma única linha limpa
for i, texto in enumerate(chunks_encontrados, 1):
    texto_linha_unica = texto.replace('\n', ' ')
    print(f"[{i}] {texto_linha_unica}")

[1] code: int class Point: coords: Tuple[int, int] label: str = '<unknown>' .. code-block:: :class: bad # Wrong: code:int # No space after colon code : int # Space before colon class Test: result: int=0 # No spaces around equality sign - Although the :pep:`526` is accepted for Python 3.6, the variable annotation syntax is the preferred syntax for stub files on all versions of Python (see :pep:`484` for details). .. rubric:: Footnotes .. [#fn-hi] *Hanging indentation* is a type-setting style where all the lines in a paragraph are indented except the first line. In the context of Python, the term is used to describe a style where the opening parenthesis of a parenthesized statement is the last non-whitespace character of the line, with subsequent lines being indented until the closing parenthesis. References ========== .. [2] Barry's GNU Mailman style guide http://barry.warsaw.us/software/STYLEGUIDE.txt .. [3] Donald Knuth's *The TeXBook*, pages 195 and 196. .. [4] http://www.wikipedia.c

In [26]:
%%time
I_IndexIVFPQ = flat.metrica(3)

[[1828 8453 5361 4218]]
CPU times: user 8.21 s, sys: 518 ms, total: 8.73 s
Wall time: 5.15 s


In [27]:
# Pega a lista de índices da query
indices_retornados = I_IndexIVFPQ[0]

# Mapeia diretamente na sua lista de chunks lida do JSON
chunks_encontrados = [textos[idx] for idx in indices_retornados if idx != -1]

# Imprime cada resultado em uma única linha limpa
for i, texto in enumerate(chunks_encontrados, 1):
    texto_linha_unica = texto.replace('\n', ' ')
    print(f"[{i}] {texto_linha_unica}")

[1] Abstract ======== This PEP describes an interpretation of multiline string constants for Python. It suggests stripping spaces after newlines and stripping a newline if it is first character after an opening quotation. Rationale ========= This PEP proposes an interpretation of multiline string constants in Python. Currently, the value of string constant is all the text between quotations, maybe with escape sequences substituted, e.g.:: def f(): """ la-la-la limona, banana """ def g(): return "This is \ string" print repr(f.__doc__) print repr(g()) prints:: '\n\tla-la-la\n\tlimona, banana\n\t' 'This is \tstring' This PEP suggest two things: - ignore the first character after opening quotation, if it is newline - ignore in string constants all spaces and tabs up to first non-whitespace character, but no more than current indentation. After applying this, previous program will print:: 'la-la-la\nlimona, banana\n' 'This is string' To get this result, previous programs could be rewritten